# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print("Dataset Title:", metadata_json['name'])
print("Description:", metadata_json['description'])
print("Identifier:", metadata_json['identifier'])
print("License:", metadata_json['license'])
print("Authors (IDs):", [author['@id'] for author in metadata_json.get('author', [])])


## 2. Data Overview
Review available record sets, fields, and their IDs. 
Croissant schemas reference entities via `@id`, which uniquely identifies each record set, field, or column.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.record_sets

available_record_set_ids = [rs['@id'] for rs in record_sets]
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - Name: {rs.get('name', 'N/A')} | @id: {rs['@id']}")

# For each record set, list its fields & columns
for rs in record_sets:
    print(f"\nRecord Set: {rs.get('name', 'N/A')} (@id: {rs['@id']})")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"    Field: {field.get('name', 'N/A')} (@id: {field['@id']})")
        columns = field.get('columns', [])
        for col in columns:
            print(f"      Column: {col.get('name', 'N/A')} (@id: {col['@id']})")

# Show sample records from the first record set
if available_record_set_ids:
    first_rs_id = available_record_set_ids[0]
    print(f"\nSample records from {first_rs_id}:")
    for i, x in enumerate(dataset.records(record_set=first_rs_id)):
        print(x)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for structured analysis. 
All record sets, fields, and columns are referenced by their `@id`.


In [ ]:
# Extract data from each record set
dataframes = {}
print("Loading all record sets by @id:")
for rs_id in available_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Record Set: {rs_id}")
    print(f"Columns (@id): {list(dataframes[rs_id].columns)}")
    print(dataframes[rs_id].head(), '\n')

# For demonstration, select the first record set for downstream steps
selected_record_set_id = available_record_set_ids[0] if available_record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing fields, categorizing, grouping.

**Example below** uses `@id` for both fields and columns. Adjust as per actual column names in the selected dataset.

In [ ]:
# Identify numeric field @id for analysis
# We'll use the first numeric column from the selected record set.
numeric_field_id = None
group_field_id = None

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Detect numeric columns by dtype -- the true @id is preserved as column name
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    # Attempt to find a categorical/grouping field
    group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and 'phenotype' in str(col).lower()]
    if group_candidates:
        group_field_id = group_candidates[0]

    if numeric_field_id:
        print(f"Selected numeric field for filtering: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if available
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Group by field {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please check dataset columns.")


## 5. Visualization
Visualize distributions or relationships between columns using matplotlib/seaborn.
Fields are referenced by `@id` (as column headers).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[selected_record_set_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of Numeric Field (@id): {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id is available, show boxplot
    if group_field_id:
        plt.figure(figsize=(7, 4))
        sns.boxplot(
            x=dataframes[selected_record_set_id][group_field_id],
            y=dataframes[selected_record_set_id][numeric_field_id]
        )
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing a Croissant-formatted FAIR^2 dataset using the `mlcroissant` library.

Key Points:
- All entities (record sets, fields, columns) are referenced and processed by their `@id`, providing consistent and reproducible access.
- Loaded record sets into DataFrames for tabular operations and visualization.
- Applied simple filtering, normalization, and grouping, referencing the precise column `@id`s.
- Visualized numeric field distributions and grouped comparisons.

Due to the small dataset size and the scope, the analysis is mainly descriptive and suitable for exploratory clinical and genotype-phenotype investigation.
